# Davinci — Human Preference Dataset Builder

Prepare a provenance-preserving preference dataset for DPO. Input records must be real comparisons reviewed by humans or a documented evaluation process. This notebook validates the schema and exact duplicates; it does not generate preferences or decide which answer is better.

The exported manifest is deliberately **not approved** by default. A human must review source provenance, license terms, privacy, consent, and intended use before changing the approval fields.

## 1. Install dependencies

In [ ]:
!pip -q install tqdm

In [ ]:
import json, hashlib, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR=Path('/content/Davinci')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci.preference import validate_preference_pair
from src.davinci.data_pipeline import sha256_file

ROOT=Path('/content/drive/MyDrive/Davinci/preference_data')
RAW=ROOT/'raw'/'preferences_raw.jsonl'
OUT=ROOT/'processed'
OUT.mkdir(parents=True,exist_ok=True)
OUTPUT_JSONL=OUT/'preferences.jsonl'
OUTPUT_MANIFEST=OUT/'preferences_manifest.json'
if not RAW.is_file(): raise FileNotFoundError(f'Add your reviewed raw preference JSONL here: {RAW}')

## 2. Declare source provenance and review status

Use an immutable source revision, such as a dataset commit, versioned export ID, or source snapshot hash. Do not leave placeholder metadata in a released dataset card.

In [ ]:
DATASET_ID=''  # Required: actual dataset/source identity.
DATASET_REVISION=''  # Required: immutable source snapshot/version.
SOURCE_DESCRIPTION=''  # Required: describe collection and annotator process.
LICENSE_REVIEWED=False
LICENSE_APPROVED_FOR_USE=False
LICENSE_REVIEW_NOTES='Pending: review provenance, upstream terms, commercial use, redistribution, privacy, and consent.'
if not DATASET_ID.strip() or not DATASET_REVISION.strip() or not SOURCE_DESCRIPTION.strip():
    raise ValueError('Set DATASET_ID, immutable DATASET_REVISION, and SOURCE_DESCRIPTION before exporting.')
if LICENSE_APPROVED_FOR_USE and not LICENSE_REVIEWED:
    raise ValueError('Cannot approve use before recording an explicit license review.')

## 3. Validate, normalize, and deduplicate comparisons

Expected JSONL schema: `{"prompt":"...","chosen":"...","rejected":"...","pair_id":"optional"}`. Empty values and identical chosen/rejected responses are rejected. Exact duplicates are removed; near-duplicate prompts and semantic quality still need separate review.

In [ ]:
valid=[]; rejected=[]; seen=set()
for line_number,line in enumerate(RAW.read_text(encoding='utf-8').splitlines(),1):
    if not line.strip(): continue
    try:
        record=json.loads(line)
        pair=validate_preference_pair(record)
        key=hashlib.sha256(json.dumps(pair,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
        if key in seen:
            rejected.append({'line':line_number,'reason':'exact_duplicate'})
            continue
        seen.add(key)
        valid.append(pair)
    except Exception as exc:
        rejected.append({'line':line_number,'reason':str(exc)})
if not valid: raise ValueError('No valid preference pairs found.')
print({'valid_pairs':len(valid),'rejected':len(rejected)})
for item in valid[:5]: print(json.dumps(item,ensure_ascii=False))

## 4. Export data, hashes, and manifest

In [ ]:
with OUTPUT_JSONL.open('w',encoding='utf-8') as handle:
    for item in valid:
        handle.write(json.dumps(item,ensure_ascii=False)+'\n')
REJECTED=OUT/'rejected_rows.json'
REJECTED.write_text(json.dumps(rejected,ensure_ascii=False,indent=2),encoding='utf-8')
manifest={
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'dataset_id':DATASET_ID,
    'dataset_revision':DATASET_REVISION,
    'purpose':'preference',
    'source_description':SOURCE_DESCRIPTION,
    'license_reviewed_by_user':LICENSE_REVIEWED,
    'license_approved_for_use':LICENSE_APPROVED_FOR_USE,
    'license_review_notes':LICENSE_REVIEW_NOTES,
    'input_file_sha256':sha256_file(RAW),
    'preference_jsonl_sha256':sha256_file(OUTPUT_JSONL),
    'valid_pairs':len(valid),
    'rejected_pairs':len(rejected),
    'deduplication':'Exact SHA-256 over canonical prompt/chosen/rejected objects',
    'known_limits':['No near-duplicate detection','No automatic correctness verification','Human preference quality must be reviewed'],
}
OUTPUT_MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Exported:',OUTPUT_JSONL,OUTPUT_MANIFEST,REJECTED,sep='\n')
print('IMPORTANT: manifest remains unapproved unless the review booleans and notes are deliberately updated after review.')

## 5. Review checklist

Before DPO training:

- Verify who created each comparison and how preferences were collected.
- Check dataset license and every upstream component's terms.
- Review personal data, consent, and sensitive content handling.
- Inspect Persian, English, and mixed-language samples separately.
- Check prompt duplication and train/validation leakage; the DPO notebook groups comparisons by normalized prompt.
- Only then set `LICENSE_REVIEWED=True`, set `LICENSE_APPROVED_FOR_USE=True` if permitted, write meaningful review notes, and rerun the export cell.

No preference dataset is included or fabricated by this repository.